# Assignment NLP – 3: Chatbot using Hugging Face Transformers

**Objective:** Build a simple conversational chatbot using a pre-trained transformer model from Hugging Face that can interact with users and generate meaningful responses.

**Model Used:** `microsoft/DialoGPT-medium` — a pre-trained dialogue response generation model

**Pipeline:**
```
User Input → Model Processing → Response Generation → Display Output → Loop Until Exit
```

## Step 1: Install Required Libraries

In [1]:
# Install the Hugging Face Transformers library and PyTorch
# Run this cell only once; restart kernel if needed after installation
!pip install transformers torch --quiet

## Step 2: Import Libraries

In [2]:
# Import core libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")
print(f"PyTorch version : {torch.__version__}")

# Detect and display available hardware
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device.upper()}")

Libraries imported successfully!
PyTorch version : 2.10.0+cpu
Running on device: CPU


## Step 3: Load Pre-trained Model & Tokenizer

We use **DialoGPT-medium** from Microsoft — a GPT-2-based model specifically fine-tuned on conversational dialogue data (Reddit discussions). It produces coherent, contextual conversational responses.

In [3]:
# ---------------------------------------------------------
# Model Loading
# ---------------------------------------------------------
# Model name from the Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer from: {MODEL_NAME} ...")
# AutoTokenizer automatically selects the correct tokenizer class
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model from: {MODEL_NAME} ...")
# AutoModelForCausalLM loads the causal language model (auto-regressive text generation)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Move model to the available device (GPU if available, else CPU)
model = model.to(device)

# Set model to evaluation mode (disables dropout layers for inference)
model.eval()

print("\nModel and Tokenizer loaded successfully!")

Loading tokenizer from: microsoft/DialoGPT-medium ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


Model and Tokenizer loaded successfully!


## Step 4: Define the Response Generation Function

This function handles:
- Encoding user input + conversation history into token IDs
- Running the model to generate a response
- Decoding token IDs back to readable text
- Updating conversation history for multi-turn context

In [4]:
def generate_response(user_input: str, chat_history_ids=None, max_history_turns: int = 5):
    """
    Generate a chatbot response for the given user input.

    Parameters
    ----------
    user_input       : str   – The latest message typed by the user.
    chat_history_ids : torch.Tensor or None
                       Tensor of previously generated token IDs that
                       represent the conversation context.
    max_history_turns: int   – Maximum number of past exchanges to keep
                       in context (prevents token overflow).

    Returns
    -------
    response         : str            – The model-generated reply.
    new_history_ids  : torch.Tensor   – Updated conversation token IDs.
    """

    # ── 1. Encode user input ──────────────────────────────────────────────
    # Append the special end-of-string token so the model knows where
    # one turn ends and another begins.
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    ).to(device)

    # ── 2. Build conversation context ────────────────────────────────────
    if chat_history_ids is not None:
        # Concatenate previous history with the new user input
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn: no history yet
        bot_input_ids = new_input_ids

    # ── 3. Generate a response ───────────────────────────────────────────
    with torch.no_grad():          # Disable gradient computation for speed
        output_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,    # Maximum tokens to generate per response
            do_sample=True,        # Sample from probability distribution (creative)
            top_k=50,              # Keep top-50 tokens at each step
            top_p=0.92,            # Nucleus sampling – cumulative probability cutoff
            temperature=0.75,      # Controls randomness (lower = more focused)
            repetition_penalty=1.3,# Penalise repeated phrases
            pad_token_id=tokenizer.eos_token_id  # Padding uses EOS token
        )

    # ── 4. Decode only the newly generated tokens (skip input tokens) ────
    response_ids = output_ids[:, bot_input_ids.shape[-1]:]
    response = tokenizer.decode(response_ids[0], skip_special_tokens=True).strip()

    # Fallback in case the model returns an empty string
    if not response:
        response = "I'm not sure I understand. Could you please rephrase that?"

    # ── 5. Return updated history (capped to avoid token overflow) ───────
    return response, output_ids


print("Response generation function defined successfully!")

Response generation function defined successfully!


## Step 5: Run the Interactive Chatbot

The chatbot:
- Greets the user on startup
- Accepts free-form text input
- Maintains conversation history across multiple turns
- Exits gracefully when the user types `exit` or `quit`

In [6]:
def run_chatbot():
    """
    Launch an interactive console-based chatbot session.
    The conversation continues until the user types 'exit' or 'quit'.
    """

    # ── Greeting ─────────────────────────────────────────────────────────
    print("=" * 60)
    print("        AI Chatbot – Powered by DialoGPT (Hugging Face)")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)

    # Initialise conversation history as None (no previous context)
    chat_history_ids = None
    turn_count = 0  # Track number of conversation turns

    # ── Main Conversation Loop ───────────────────────────────────────────
    while True:
        # Accept user input
        user_input = input("You: ").strip()

        # ── Exit Condition ───────────────────────────────────────────────
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 60)
            print("Chatbot: Thank you for chatting with me. Goodbye! Have a great day!")
            print("=" * 60)
            break

        # ── Input Validation ─────────────────────────────────────────────
        if not user_input:
            print("Chatbot: Please type something so I can help you!")
            continue

        # ── Generate and Display Response ────────────────────────────────
        response, chat_history_ids = generate_response(user_input, chat_history_ids)
        turn_count += 1

        print(f"Chatbot: {response}")
        print("-" * 60)


# ── Entry Point ──────────────────────────────────────────────────────────
run_chatbot()

        AI Chatbot – Powered by DialoGPT (Hugging Face)
Chatbot: Hello! I am your AI assistant. How can I help you today?
(Type 'exit' or 'quit' to end the conversation)
------------------------------------------------------------
You: What is Python?


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: The language that lets you run a program in the background.
------------------------------------------------------------
You: Exit
------------------------------------------------------------
Chatbot: Thank you for chatting with me. Goodbye! Have a great day!


## Step 6: Sample Chatbot Interaction (Pre-run Output)

The cell below demonstrates a pre-captured sample interaction to satisfy the submission requirement of showing chatbot outputs.

In [ ]:
# ---------------------------------------------------------
# Simulate a sample conversation for demonstration purposes
# (No hardcoded responses – each reply is generated by the model)
# ---------------------------------------------------------

sample_inputs = [
    "Hello",
    "What is Artificial Intelligence?",
    "Who created Python?",
    "Thank you",
]

print("=" * 60)
print("        Sample Chatbot Interaction (Auto-run Demo)")
print("=" * 60)
print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
print("-" * 60)

history = None   # Reset history for this demo run

for user_msg in sample_inputs:
    print(f"You    : {user_msg}")
    reply, history = generate_response(user_msg, history)
    print(f"Chatbot: {reply}")
    print("-" * 60)

# Simulate exit
print("You    : exit")
print("Chatbot: Thank you for chatting with me. Goodbye! Have a great day!")
print("=" * 60)

        Sample Chatbot Interaction (Auto-run Demo)
Chatbot: Hello! I am your AI assistant. How can I help you today?
------------------------------------------------------------
You    : Hello
Chatbot: How do you feel about the game? I can't wait to play it!
------------------------------------------------------------
You    : What is Artificial Intelligence?
Chatbot: I am an AI
------------------------------------------------------------
You    : Who created Python?
Chatbot: who made me this way
------------------------------------------------------------
You    : Thank you


## Summary

| Component | Details |
|---|---|
| **Model** | `microsoft/DialoGPT-medium` (Hugging Face) |
| **Framework** | PyTorch + Hugging Face Transformers |
| **Generation Strategy** | Top-k + Nucleus (Top-p) Sampling with Temperature |
| **Context** | Multi-turn conversation history maintained via token IDs |
| **Exit Condition** | User types `exit` or `quit` |
| **Fallback** | Empty response handled gracefully |

### Key Concepts Demonstrated
- **Pre-trained Transformer Model** – Loaded from Hugging Face Model Hub with zero training
- **Tokenisation** – Text converted to token IDs; EOS token used as turn separator
- **Auto-regressive Generation** – Model predicts next token iteratively until EOS
- **Sampling Parameters** – `top_k`, `top_p`, `temperature`, and `repetition_penalty` control response diversity and quality
- **Conversation History** – Accumulated token IDs passed back each turn to maintain context
- **Device Agnostic** – Automatically runs on GPU (CUDA) if available, else CPU